# display version of python used 

In [40]:
!py --version

Python 3.10.8


# import the libary

In [41]:
from paddleocr import PaddleOCR 
import paddle
import cv2
import numpy as np
import os
from llama_cpp import Llama
import json
import onnxruntime as ort
import onnx
import math

# device used

In [42]:
paddle.device.get_device()

'cpu'

In [43]:
paddle.device.cuda.device_count()

0

# loading image path

In [44]:
i="1"
image_path = f"D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/{i}.jpg"
image_path

'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/1.jpg'

# read img

In [45]:
print(os.path.exists(image_path))

True


In [46]:
img = cv2.imread(image_path)

# onnx

In [47]:
det_path="D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/ai_model/det_onnx_model/inference.onnx"

In [48]:
# # import cv2
# # import numpy as np
# # import onnxruntime as ort

# sess = ort.InferenceSession(det_path, providers=["CPUExecutionProvider"])

# for inp in sess.get_inputs():
#     print("name:", inp.name)
#     print("shape:", inp.shape)
#     print("dtype:", inp.type)

In [49]:
# def preprocess_det(img, target_size=960):
#     """
#     Resize image proportionally (preserving aspect ratio), 
#     then pad to make both height and width multiples of 32.
#     """
#     h, w = img.shape[:2]

#     # Step 1: Scale so the longer side becomes target_size
#     scale = target_size / max(h, w)
#     new_w = int(w * scale)
#     new_h = int(h * scale)
#     resized = cv2.resize(img, (new_w, new_h))

#     # Step 2: Round height and width UP to the nearest multiple of 32
#     padded_h = math.ceil(resized.shape[0] / 32) * 32 
#     padded_w = math.ceil(resized.shape[1] / 32) * 32 
#     # padded_h = (resized.shape[0] + 31) // 32 * 32
#     # padded_w = (resized.shape[1] + 31) // 32 * 32

#     # Step 3: Create a black canvas of the padded size
#     padded = np.zeros((padded_h, padded_w, 3), dtype=np.uint8)

#     # Step 4: Paste the resized image into the top-left corner
#     padded[:resized.shape[0], :resized.shape[1], :] = resized

#     # Step 5: Normalize and convert to model input format (NCHW)
#     img_norm = padded.astype(np.float32) / 255.0
#     img_norm = img_norm.transpose(2, 0, 1)        # HWC -> CHW
#     img_norm = np.expand_dims(img_norm, axis=0)   # add batch dim -> (1,3,H,W)

#     return img_norm.astype(np.float32), scale


# # # --- Usage ---
# # img = cv2.imread("invoice.jpg")
# det_input, scale = preprocess_det(img, target_size=960)

# # 4. Run inference
# output=session.run(None, {"x": det_input})

# # 5. See the result
# print("Output shape:", output[0].shape)
# print(output)

In [50]:
# heatmap = output[0] # extract the (960, 960) 2D array

# print("Min:", heatmap.min())
# print("Max:", heatmap.max())
# print("Mean:", heatmap.mean())
# print("Values > 0.3:", np.sum(heatmap > 0.3))  # count how many pixels look like text
# print("Values > 0.5:", np.sum(heatmap > 0.5))

In [51]:
# heatmap = output[0][0,0]
# thresh=0.6
# heatmap_normalized = (heatmap>thresh).astype(np.uint8)*255
# cv2.imwrite("heatmap_check_thresh.png", heatmap_normalized)

# extracting the lines from the img 

In [52]:
# if (img is None) :
#     print(f"{image_path} this is incorrect no img found")
# else:
   
#     # converting img to gray
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#     # cv2.imwrite(f'./output/opencv/gray{i}.jpeg', gray)

#     # splits the image and clahe 
#     # C – Contrast
#     # L – Limited
#     # A – Adaptive
#     # H – Histogram
#     # E – Equalization
#     clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
#     gray_clahe = clahe.apply(gray)    
#     # cv2.imwrite(f'./output/opencv/gray_clahe{i}.jpeg', gray_clahe)
    
#     # Better threshold (adaptive handles uneven lighting)
#     thresh = cv2.adaptiveThreshold(
#         gray_clahe, 255,
#         cv2.ADAPTIVE_THRESH_MEAN_C,
#         cv2.THRESH_BINARY_INV,
#         15, 10
#     )
     
    
#     # --- Horizontal lines ---
#     horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (60,1))
#     horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
#     cv2.imwrite(f'./output/opencv/horizontal{i}.jpeg', horizontal)
    
#     # connect broken horizontal lines
#     horizontal = cv2.dilate(horizontal, np.ones((2,2),np.uint8), iterations=3)
#     cv2.imwrite(f'./output/opencv/horizontal_dilated{i}.jpeg', horizontal)
    
#     # --- Vertical lines ---
#     vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,60))
#     vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
#     # cv2.imwrite(f'./output/opencv/vertical{i}.jpeg', vertical)
     
#     # connect broken vertical lines
#     vertical = cv2.dilate(vertical, np.ones((2,2),np.uint8), iterations=3)
    
#     cv2.imwrite(f'./output/opencv/vertical_dilate{i}.jpeg', vertical)
#     h_img_path=f'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/output/opencv/horizontal_dilated{i}.jpeg'
#     h_img = cv2.imread(h_img_path)

In [296]:
def get_vh_lines(img,image_path):
    if (img is None) :
        print(f"{image_path} this is incorrect no img found")
    else:
       
        # converting img to gray
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        # cv2.imwrite(f'./output/opencv/gray{i}.jpeg', gray)
    
        # splits the image and clahe 
        # C – Contrast
        # L – Limited
        # A – Adaptive
        # H – Histogram
        # E – Equalization
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        gray_clahe = clahe.apply(gray)    
        # cv2.imwrite(f'./output/opencv/gray_clahe{i}.jpeg', gray_clahe)
        
        # Better threshold (adaptive handles uneven lighting)
        thresh = cv2.adaptiveThreshold(
            gray_clahe, 255,
            cv2.ADAPTIVE_THRESH_MEAN_C,
            cv2.THRESH_BINARY_INV,
            15, 10
        )
         
        
        # --- Horizontal lines ---
        horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (60,1))
        horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
        # cv2.imwrite(f'./output/opencv/horizontal{i}.jpeg', horizontal)
        
        # connect broken horizontal lines
        horizontal = cv2.dilate(horizontal, np.ones((2,2),np.uint8), iterations=3)
        # cv2.imwrite(f'./output/opencv/horizontal_dilated{i}.jpeg', horizontal)
        
        # --- Vertical lines ---
        vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,60))
        vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
        # cv2.imwrite(f'./output/opencv/vertical{i}.jpeg', vertical)
         
        # connect broken vertical lines
        vertical = cv2.dilate(vertical, np.ones((2,2),np.uint8), iterations=3)
        
        # cv2.imwrite(f'./output/opencv/vertical_dilate{i}.jpeg', vertical)
        # h_img_path=f'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/output/opencv/horizontal_dilated{i}.jpeg'
        # h_img = cv2.imread(h_img_path)

        return horizontal,vertical

def get_h_counters(horizontal):
    contours_h, _ = cv2.findContours(horizontal, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return contours_h
def get_v_counters(vertical):
    contours_v, _ = cv2.findContours(vertical, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return contours_v

def get_h_angle(contours_h):
    angle_arr=[]
    for counter in contours_h:
        hx, hy, x0, y0 = cv2.fitLine(counter, cv2.DIST_L2, 0, 0.01, 0.01)
        angle = np.arctan2(hy, hx)
        rotation_angle =np.degrees(angle)
        print(rotation_angle)
        if int(rotation_angle) == 0:
            continue 
        angle_arr.append(rotation_angle)
    return angle_arr
    
def get_v_angle(contours_v):
    angle_arr=[]
    for counter in contours_v:
        vx, vy, x0, y0 = cv2.fitLine(counter, cv2.DIST_L2, 0, 0.01, 0.01)
        angle = np.arctan(vx, vy)
        # print(angle)
        rotation_angle =np.degrees(-angle)
        if int(rotation_angle) == 0:
            continue  
        angle_arr.append(rotation_angle)
    return angle_arr

def rotate_angle_avg(angle_arr):
    rotation_angle_avg=float(sum(angle_arr)/len(angle_arr))
    print(rotation_angle_avg)
    return rotation_angle_avg

def rotate_img(img,rotation_angle_avg):
    (h, w) = img.shape[:2]
    center = (w // 2, h // 2)
    img_matrix = cv2.getRotationMatrix2D(center, rotation_angle_avg, 1.0)
    rotated_img = cv2.warpAffine(img, img_matrix, (w, h))
    return rotated_img


    
    

In [297]:
def rotation_correct_img(img,image_path):
    horizontal,vertical=get_vh_lines(img,image_path)
    if len(horizontal) >= len(vertical):
        contours_h=get_h_counters(horizontal)
        angle_arr_h=get_h_angle(contours_h)
        if len(angle_arr_h) == 0:
            return img
        rotation_angle_avg=rotate_angle_avg(angle_arr_h)
        rotated_img=rotate_img(img,rotation_angle_avg)
        return rotated_img
    elif len(horizontal) < len(vertical):
        contours_v=get_v_counters(vertical)
        angle_arr_v=get_v_angle(contours_v)
        if len(angle_arr_v) == 0:
            return img
        rotation_angle_avg=rotate_angle_avg(angle_arr_v)
        rotated_img=rotate_img(img,-rotation_angle_avg)
        return rotated_img
    elif len(horizontal) == 0 or len(vertical):
        return img


In [298]:
# rotated_img=rotation_correct_img(img,image_path)
# cv2.imwrite(f'./output/opencv/rotated_img_final-----{i}.jpeg', rotated_img)

In [299]:
# angle_arr=[]
# for contour in contours:
#     vx, vy, x0, y0 = cv2.fitLine(contour, cv2.DIST_L2, 0, 0.01, 0.01)
#     angle = np.arctan2(vy, vx)
#     rotation_angle =np.degrees(angle)
#     angle_arr.append(rotation_angle)
    
# angel_avg=sum(angle_arr)/len(angle_arr)
# angel_avg

In [300]:
#  # Find contours
# contours_h, _ = cv2.findContours(horizontal, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
# contours_v, _ = cv2.findContours(vertical, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# # result = cv2.drawContours(h_img, contours[1], -1, (0, 255, 0), 2)

# # cv2.imwrite(f'./output/opencv/output---{i}.jpeg', result)

In [301]:
# len(contours[1])

In [302]:
  # vxh, vyh, x0h, y0h = cv2.fitLine(contours_h[1], cv2.DIST_L2, 0, 0.01, 0.01)
  # vxv, vyv, x0v, y0v = cv2.fitLine(contours_v[6], cv2.DIST_L2, 0, 0.01, 0.01)

In [303]:
# angle_arr

In [304]:
# # angle = np.arctan2(vy, vx)
# angle = np.arctan(vxv, vyv)
# # angle = np.arctan2(vyv, vxv)
# angle

In [305]:
# rotation_angle =float( np.degrees(angle))
# rotation_angle

In [306]:
# (h, w) = img.shape[:2]
# center = (w // 2, h // 2)

# M = cv2.getRotationMatrix2D(center, rotation_angle, 1.0)
# M = cv2.getRotationMatrix2D(center, -rotation_angle, 1.0) #1.0 no change to img is made
# rotated = cv2.warpAffine(img, M, (w, h))#m tell how to move the points 

# cv2.imwrite(f'./output/opencv/rotated{i}.jpeg', rotated)

In [307]:
# contours_arr = list(contours)
# contours_arr

In [308]:
# contours_x=sorted(contours_arr[1], key=lambda p: p[0][0])
# contours_y=sorted(contours_arr[1], key=lambda p: p[0][1])


In [309]:
# for counter_pnt  in contours_x:
    

In [310]:
# contours_arr[1]

In [311]:
# contours_x

In [312]:
# contours_y

In [313]:
# for i in contours[1].tolist():
#     print(i)

In [314]:
# contours[1].tolist()

In [315]:
# contoursssss, _ = cv2.findContours(horizontal, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

In [316]:
# len(contoursssss[0])

In [317]:
# contoursssss[0]

In [318]:
# contours[1].tolist()

# run paddleocr on img 

In [ ]:
if (img is None) :
    print(f"{image_path} this is incorrect no img found")
else:
    # setting tread 
    # os.environ["OMP_NUM_THREADS"] = "8"
    # os.environ["MKL_NUM_THREADS"] = "8"
    # applying ocr on the image
    ocr = PaddleOCR(use_doc_orientation_classify=False, 
        use_doc_unwarping=False, 
        use_textline_orientation=False,
        lang='en',
        device=paddle.device.get_device(),
        cpu_threads=8 
       )
    result = ocr.predict(rotation_correct_img(img,image_path))

Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


[0.02888323]
[0.05407808]
[0.01889551]
[0.11438435]
[0.36477593]
[0.40443003]


C:\Users\Anuj S Jain\AppData\Local\Temp\ipykernel_14620\204788707.py:66: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  if int(rotation_angle) == 0:


# find and draw the counter

In [ ]:
# output = img.copy()

# # for index,cnt in enumerate(contours):
# #     x,y,w,h = cv2.boundingRect(cnt)
# #     # filter noise
# #     if w > 100 and h > 100:
# cv2.drawContours(output, contours, -1, (0,255,0), 1)
# cv2.imwrite(f'./output/opencv/output{i}.jpeg', output)

# angle correction of ocr box

# start from here 

In [ ]:
import numpy as np
import math


# -----------------------------
# 1. GET ANGLE FROM ONE BOX
# -----------------------------
def get_angle_from_box(box):

    # box = 4 points [[x,y],...]

    (x1, y1), (x2, y2) = box[0], box[1]

    dx = x2 - x1
    dy = y2 - y1

    angle = math.degrees(math.atan2(dy, dx))

    return angle


# -----------------------------
# 2. GET GLOBAL ANGLE (AVERAGE)
# -----------------------------
def get_global_angle(ocr_boxes):

    angles = []

    # for box, text, score in ocr_boxes:
    #     angles.append(get_angle_from_box(box))
    
    for box in ocr_boxes:
        angles.append(get_angle_from_box(box))

    return np.median(angles)  # robust against noise


# -----------------------------
# 3. ROTATE POINT
# -----------------------------
def rotate_point(x, y, cx, cy, angle):

    theta = math.radians(-angle)  # negative to correct

    x -= cx
    y -= cy

    xr = x * math.cos(theta) - y * math.sin(theta)
    yr = x * math.sin(theta) + y * math.cos(theta)

    return xr + cx, yr + cy


# -----------------------------
# 4. ROTATE POLYGON
# -----------------------------
def rotate_box(box, cx, cy, angle):

    new_box = []

    for x, y in box:
        nx, ny = rotate_point(x, y, cx, cy, angle)
        new_box.append([nx, ny])

    return new_box


# -----------------------------
# 5. FIX ALL OCR BOXES
# -----------------------------
def deskew_ocr_boxes(ocr_boxes, image_shape):

    h, w = image_shape[:2]

    cx, cy = w / 2, h / 2

    # step 1: find global angle
    angle = get_global_angle(ocr_boxes)

    print("Detected angle:", angle)

    # step 2: rotate all boxes
    corrected = []

    # for box, text, score in ocr_boxes:
    #     new_box = rotate_box(box, cx, cy, angle)
    #     corrected.append((new_box, text, score))
    
    for box in ocr_boxes:
        new_box = rotate_box(box, cx, cy, angle)
        corrected.append(new_box)

    return corrected, angle

# Downscaling the paddle ocr border to small

In [ ]:
def downscaling(poly_list):
    poly_downscaling=[]
    for poly in poly_list:
        # convert to float for calculation
        poly = np.array(poly, dtype=np.float32)
        # cv2.polylines(output, [np.array(poly, dtype=np.int32)],isClosed=True, color=(255, 0, 0),thickness=2) 
    
        x_min, y_min = np.min(poly, axis=0)
        x_max, y_max = np.max(poly, axis=0)
        w = x_max - x_min
        h = y_max - y_min
    
        # adaptive shrink
        shrink_factor_h = max(0.6, min(0.9, 1 - 20/w))
        shrink_factor_v = max(0.6, min(0.9, 1 - 20/h))
        # print(poly)
        
        # compute center
        center_x = np.mean(poly[:, 0])
        center_y = np.mean(poly[:, 1])
    
        poly_box=[]
        for (x,y) in poly:
            x_new=int(center_x+(x-center_x)*shrink_factor_h)
            y_new=int(center_y+(y-center_y)*shrink_factor_v)
            poly_box.append([x_new,y_new])
        # cv2.polylines(output, [np.array(poly_box, dtype=np.int32)],isClosed=True, color=(0, 255, 255),thickness=2) 
        # cv2.imwrite(f'./output/opencv/scaling{i}.jpeg', output)
        poly_downscaling.append(np.array(poly_box, dtype=np.int16))
    return poly_downscaling


# make an array by combining the ploy points , text , and confident scores

In [77]:
result[0]['rec_texts']

NameError: name 'result' is not defined

In [78]:
result[0]['dt_polys'][0]

NameError: name 'result' is not defined

In [43]:
downscaling_poly=downscaling(result[0]['dt_polys'])
ocr_result_rt=deskew_ocr_boxes(downscaling_poly,img.shape)
ocr_result= [(poly , text , scores)
    for poly , text , scores  in 
        zip(ocr_result_rt[0],
            result[0]['rec_texts'],
            result[0]['rec_scores'])]


Detected angle: 0.0


# arranging the text box of paddleocr horizontally

In [44]:
def get_center(box):
    #[[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    x_coords = [p[0] for p in box]
    y_coords = [p[1] for p in box]
    return sum(x_coords) / 4, sum(y_coords) / 4


def group_boxes_into_lines(boxes, y_threshold=10):
    # Step 1: compute centers
    box_centers = [(box, get_center(box),text,score) for box,text,score in boxes]

    # Step 2: sort by Y (top to bottom)
    box_centers.sort(key=lambda b: b[1][1])

    lines = []
    current_line = []
    for box, (cx, cy) ,text ,score in box_centers:
        if score >= 0.9:
            if not current_line:
                current_line.append((box, cx, cy, text, score))
                continue
    
            _, _, prev_cy, _, _ = current_line[-1]
    
            # Step 3: check if same line
            if abs(cy - prev_cy) < y_threshold:
                current_line.append((box, cx, cy, text, score))
            else:
                lines.append(current_line)
                current_line = [(box, cx, cy, text, score)]
        else:
            continue

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line by X (left to right)
    sorted_lines = []
    sorted_lines_text = []
    for line in lines:
        line.sort(key=lambda b: b[1])  # sort by center x
        sorted_lines.append([b[0] for b in line])
        sorted_lines_text.append([b[3] for b in line])

    return sorted_lines,sorted_lines_text

In [45]:
poly,text = group_boxes_into_lines(ocr_result)

In [51]:
len(text)

34

# llm integration

In [46]:
# chr(10).join( result[0]['rec_texts'])

In [47]:
# chr(10).join(text[0])

In [52]:
headers = text[0]

table = " \n"
# table += "|" + "|".join(["---"] * len(headers)) + "|\n"

for row in text[0:]:
    row = row + [""] 
    table += "| " + " | ".join(row) + " |\n"

print(len(table))
print(table)

2269
 
| PARAMESHWARA STATIONERY | Bill No. | 11,904 |  |
| HOSALINE ROAD, NEAR SANTHEPETE CIRCLE | To: | Date: | 4-Nov-21 |  |
| HASSAN-573201 |  |
| MCE CES |  |
| Ph. 08172-234823 | HASSAN |  |
| GSTIN.: 29AFHPM3915J1Z5 | GST Id. No.: |  |
| SI No Item Description | HSN | Code | MRP | Rate | Qty | Basic. | CGST | SGST | Net Amt. |  |
| % | Amt. | % | Amt. |  |
| 1 | OMEGA CONTAINER | 3926 | 160 | 120.00 | 40 Pc. | 4067.80 | 9.00 | 366.10 | 9.00 | 366.10 | 4800.00 |  |
| 2 | OMEGA DRAFT CLIP | 250 | 185.00 | 10 Pkt | 1567.80 | 9.00 | 141.10 | 9.00 | 141.10 | 1850.00 |  |
| 3 | OMEGA ROLLER SCALE | 9017 | 33 | 26.00 | 20 Pc. | 464.29 | 6.00 | 27.86 | 6.00 | 27.86 | 520.00 |  |
| 4 | DRAWING BOARD | 500 | 325.00 | 2 | Pc. | 550.85 | 9.00 | 49.58 | 9.00 | 49.58 | 650.00 |  |
| 5 | CASIO 991ES PLUS | 8470 | 1,295 | 1,085.00 | 9194.92 | 9.00 | 827.54 | 9.00 | 827.54 | 10850.00 |  |
| 6 | A4 STICK FILE | 10 | 7.50 | 100 Pc. | 669.64 | 6.00 | 40.18 | 6.00 | 40.18 | 750.00 |  |
| 7 | CENTER 

In [55]:
llm = Llama(
    model_path="llama3-Q5_K_M.gguf",
    n_ctx=2048,
    n_threads=8,
    verbose=False  
)

llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


In [56]:
# prompt = f"""You are a JSON extraction engine. Output ONLY a single valid JSON object. No markdown, no code fences, no explanation, no extra text before or after.

# TABLE FORMAT NOTES:
# - Product/item descriptions are split across multiple rows (only the first row of an item has numbers; following rows are description continuations). Merge all continuation lines into that item's item_name.
# - Ignore rows that are only headers, footers, or tax lines (CGST, SGST, VAT, GST, E&OE, HSN/SAC) — these are not line items.
# - The final total appears in a row containing "Total" or "Amount Chargeable".

# FIELD DEFINITIONS:
# - "item_name" = the name/description of the product, item, service, or goods being billed (merge multi-line description text into this single field)
# - "quantity" = the unit count, e.g. "xxxxx Nos", "xxx pcs", "x hrs"
# - "amount" = the row TOTAL value for that line (the larger number) — NOT the per-unit rate/price
# - Ignore per-unit rate/price columns when filling amount
# - For total_amount, use the final numeric grand total (e.g. "xxxxxxx.xx"), not any amount written in words

# OUTPUT SCHEMA (use exactly these fields, always as strings, use "" if missing):
# {{
#   "items": [
#     {{"item_name": "", "quantity": "", "amount": ""}}
#   ],
#   "total_amount": ""
# }}

# Now extract from this table. Return only ONE fully closed JSON object (every {{ must have a matching }}, every [ must have a matching ]), then stop.

# TABLE:
# {table}

# JSON:"""

In [64]:
prompt = f"""You are a JSON extraction engine. Output ONLY a single valid JSON object. No markdown, no code fences, no explanation, no extra text before or after.
TABLE FORMAT NOTES:
- Product/item descriptions are split across multiple rows (only the first row of an item has numbers; following rows are description continuations). Merge all continuation lines into that item's item_name.
- Ignore rows that are only headers, footers, or tax lines (CGST, SGST, VAT, GST, E&OE, HSN/SAC) — these are not line items.
- The final total appears in a row containing "Total" or "Amount Chargeable".
- The invoice number usually appears near the top, labeled "Invoice No", "Invoice #", "Bill No", "Inv No", or similar — extract only the identifier value, not the label, donnot include the label.
- The invoice date usually appears near the top, labeled "Invoice Date", "Date", "Bill Date", or "Dated" — extract only the date value, not the label. If multiple dates appear (e.g. invoice date vs due date vs delivery date), prefer the one explicitly labeled as the invoice/bill date.
- The vendor name is the name of the company/business issuing the invoice (usually at the very top, often near a logo or letterhead, and distinct from the "Bill To" / "Ship To" customer name). Do NOT confuse it with the customer/buyer name.
FIELD DEFINITIONS:
- "item_name" = the name/description of the product, item, service, or goods being billed (merge multi-line description text into this single field)
- "quantity" = the unit count, e.g. "xxxxx Nos", "xxx pcs", "x hrs"
- "amount" = the row TOTAL value for that line (the larger number) — NOT the per-unit rate/price
- Ignore per-unit rate/price columns when filling amount
- For total_amount, use the final numeric grand total (e.g. "xxxxxxx.xx"), not any amount written in words
- "invoice_number" = the invoice/bill identifier as printed (keep original formatting, e.g. "INV-2024-0091")
- "invoice_date" = the invoice date as printed on the document (original formatting, e.g. "12/04/2024" or "04-Dec-2024"), (reformat , e.g. "dd/mm/yyyy")
- "vendor_name" = the name of the issuing company/seller (not the buyer/customer)
OUTPUT SCHEMA (use exactly these fields, always as strings, use "" if missing):
{{
  "invoice_number": "",
  "invoice_date": "",
  "vendor_name": "",
  "items": [
    {{"item_name": "", "quantity": "", "amount": ""}}
  ],
  "total_amount": ""
}}
Now extract from this table. Return only ONE fully closed JSON object (every {{ must have a matching }}, every [ must have a matching ]), then stop.
TABLE:
{table}
JSON:"""

In [65]:
# prompt=ocr_input+rule
output = llm(
    prompt,
    temperature=0,
    max_tokens=len(table),
    # top_p=0.9,
    #top_k=40,
    repeat_penalty=1.05,
    # grammar=grammar,
    stop=["}  {", "} {", "}\n{", "}\n\n"]
)

In [66]:
output

{'id': 'cmpl-ef9ae216-4ec0-4ca9-8ae3-519e4cd764ea',
 'object': 'text_completion',
 'created': 1783448902,
 'model': 'llama3-Q5_K_M.gguf',
 'choices': [{'text': ' \n{\n  "invoice_number": "11,904",\n  "invoice_date": "04-Nov-21",\n  "vendor_name": "PARAMESHWARA STATIONERY",\n  "items": [\n    {\n      "item_name": "OMEGA CONTAINER",\n      "quantity": "40 Pc.",\n      "amount": "4800.00"\n    },\n    {\n      "item_name": "OMEGA DRAFT CLIP",\n      "quantity": "10 Pkt",\n      "amount": "1850.00"\n    },\n    {\n      "item_name": "OMEGA ROLLER SCALE",\n      "quantity": "20 Pc.",\n      "amount": "520.00"\n    },\n    {\n      "item_name": "DRAWING BOARD",\n      "quantity": "2 Pc.",\n      "amount": "650.00"\n    },\n    {\n      "item_name": "CASIO 991ES PLUS",\n      "quantity": "1",\n      "amount": "10850.00"\n    },\n    {\n      "item_name": "A4 STICK FILE",\n      "quantity": "100 Pc.",\n      "amount": "750.00"\n    },\n    {\n      "item_name": "CENTER FRUIT",\n      "quantit

In [67]:
import json
decoder = json.JSONDecoder()
result, _ = decoder.raw_decode(output["choices"][0]["text"].strip())
result

JSONDecodeError: Unterminated string starting at: line 38 column 19 (char 803)